# Phase 3: Local CIFAR-100 Fox–Wolf Dataset Preparation

This notebook loads your locally downloaded CIFAR-100 files, keeps only **fox** and **wolf**, creates a reproducible **80/20 train-validation split**, leaves the official test set untouched, and saves the split indices for Phase 4.

Update only `DATA_ROOT` in the configuration cell.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.__version__, torchvision.__version__)

2.11.0+cpu 0.26.0+cpu


Mounted at /content/drive


In [3]:
import os

print(os.path.exists("/content/drive/MyDrive"))

True


In [10]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "cifar-100-python" in dirs:
        print("Found at:")
        print(os.path.join(root, "cifar-100-python"))

Found at:
/content/drive/MyDrive/Datasets/cifar-100-python


KeyboardInterrupt: 

## 1. Configuration

`DATA_ROOT` must point to the folder that **contains** `cifar-100-python`.

For example, if your files are:

```text
C:/Users/Habib/Datasets/cifar-100-python/train
C:/Users/Habib/Datasets/cifar-100-python/test
C:/Users/Habib/Datasets/cifar-100-python/meta
```

use:

```python
DATA_ROOT = Path(r"C:/Users/Habib/Datasets")
```

Do not point `DATA_ROOT` directly to `cifar-100-python`.

The extra `file.txt` visible in your Kaggle folder is not used by torchvision.


In [6]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Datasets")


In [7]:
print(DATA_ROOT)
print((DATA_ROOT / "cifar-100-python").exists())

/content/drive/MyDrive/Datasets
True


In [23]:
DATA_ROOT = Path(r"/content/drive/MyDrive/Datasets")  # CHANGE THIS
OUTPUT_DIR = Path("./phase3_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_CLASSES = ["fox", "wolf"]
VALIDATION_FRACTION = 0.20
BATCH_SIZE = 64
NUM_WORKERS = 0

print("Expected folder:", (DATA_ROOT / "cifar-100-python").resolve())

Expected folder: /content/drive/MyDrive/Datasets/cifar-100-python


In [24]:
print(DATA_ROOT)

/content/drive/MyDrive/Datasets


In [25]:
import os

print(os.listdir(DATA_ROOT))

['cifar-100-python']


In [26]:
cifar_train_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False
)

cifar_test_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=False,
    download=False
)

print(len(cifar_train_full))
print(len(cifar_test_full))

50000
10000


## 2. Verify the local CIFAR-100 files

In [27]:
folder = DATA_ROOT / "cifar-100-python"
required = ["train", "test", "meta"]

if not folder.exists():
    raise FileNotFoundError(
        f"Could not find {folder.resolve()}. Change DATA_ROOT."
    )

missing = [name for name in required if not (folder / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing CIFAR-100 files: {missing}")

print("Local CIFAR-100 files found.")

Local CIFAR-100 files found.


## 3. Load CIFAR-100 without downloading

In [28]:
cifar_train_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT), train=True, download=False
)
cifar_test_full = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT), train=False, download=False
)

print("Original train:", len(cifar_train_full))
print("Original test:", len(cifar_test_full))

Original train: 50000
Original test: 10000


## 4. Identify fox and wolf labels

In [29]:
missing_classes = [
    name for name in SELECTED_CLASSES
    if name not in cifar_train_full.class_to_idx
]

if missing_classes:
    raise ValueError(
        f"These classes were not found in CIFAR-100: {missing_classes}"
    )

original_ids = {
    name: cifar_train_full.class_to_idx[name]
    for name in SELECTED_CLASSES
}

old_to_new = {
    original_ids[name]: new_id
    for new_id, name in enumerate(SELECTED_CLASSES)
}

new_to_name = {
    new_id: name
    for new_id, name in enumerate(SELECTED_CLASSES)
}

print("Original IDs:", original_ids)
print("Final mapping:", new_to_name)


Original IDs: {'fox': 34, 'wolf': 97}
Final mapping: {0: 'fox', 1: 'wolf'}


## 5. Find fox and wolf indices

In [30]:
selected_ids = set(old_to_new.keys())

all_train_indices = np.array([
    i for i, y in enumerate(cifar_train_full.targets)
    if y in selected_ids
], dtype=np.int64)

test_indices = np.array([
    i for i, y in enumerate(cifar_test_full.targets)
    if y in selected_ids
], dtype=np.int64)

all_train_original_labels = np.array([
    cifar_train_full.targets[i] for i in all_train_indices
])

print("Fox/wolf train total:", len(all_train_indices))
print("Fox/wolf test total:", len(test_indices))

Fox/wolf train total: 1000
Fox/wolf test total: 200


## 6. Create a stratified train-validation split

The validation set is created from the original CIFAR-100 training data. The official test set stays untouched.

In [31]:
train_indices, validation_indices = train_test_split(
    all_train_indices,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    shuffle=True,
    stratify=all_train_original_labels
)

train_indices = np.asarray(train_indices, dtype=np.int64)
validation_indices = np.asarray(validation_indices, dtype=np.int64)
test_indices = np.asarray(test_indices, dtype=np.int64)

# Integrity checks
assert len(set(train_indices) & set(validation_indices)) == 0, (
    "Training and validation indices overlap."
)
assert len(train_indices) + len(validation_indices) == len(all_train_indices), (
    "The training/validation split lost or duplicated images."
)
assert len(np.unique(test_indices)) == len(test_indices), (
    "Duplicate test indices were found."
)

print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))
print("The official CIFAR-100 test set remains separate and untouched.")


Train: 800
Validation: 200
Test: 200
The official CIFAR-100 test set remains separate and untouched.


## 7. Check class balance

In [32]:
def remapped_labels(dataset, indices):
    return np.array([
        old_to_new[dataset.targets[int(i)]]
        for i in indices
    ], dtype=np.int64)

train_labels = remapped_labels(cifar_train_full, train_indices)
validation_labels = remapped_labels(cifar_train_full, validation_indices)
test_labels = remapped_labels(cifar_test_full, test_indices)

def describe(name, labels):
    counts = Counter(labels.tolist())
    return {
        "split": name,
        "total": len(labels),
        "fox": counts.get(0, 0),
        "wolf": counts.get(1, 0),
    }

summary = pd.DataFrame([
    describe("Train", train_labels),
    describe("Validation", validation_labels),
    describe("Test", test_labels),
])

assert set(train_labels.tolist()) == {0, 1}
assert set(validation_labels.tolist()) == {0, 1}
assert set(test_labels.tolist()) == {0, 1}
assert summary["total"].sum() == (
    len(train_indices) + len(validation_indices) + len(test_indices)
)

summary


,split,total,fox,wolf
0,Train,800,400,400
1,Validation,200,100,100
2,Test,200,100,100


Expected distribution:

- Train: 800 images = 400 fox + 400 wolf
- Validation: 200 images = 100 fox + 100 wolf
- Test: 200 images = 100 fox + 100 wolf

## 8. Save the fixed splits for later phases

In [33]:
np.save(OUTPUT_DIR / "train_indices.npy", train_indices)
np.save(OUTPUT_DIR / "validation_indices.npy", validation_indices)
np.save(OUTPUT_DIR / "test_indices.npy", test_indices)
summary.to_csv(OUTPUT_DIR / "class_summary.csv", index=False)

metadata = {
    "data_root": str(DATA_ROOT.resolve()),
    "selected_classes": SELECTED_CLASSES,
    "new_label_to_class": {
        str(new_id): class_name
        for new_id, class_name in new_to_name.items()
    },
    "original_cifar100_ids": original_ids,
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "train_size": int(len(train_indices)),
    "validation_size": int(len(validation_indices)),
    "test_size": int(len(test_indices)),
    "split_policy": (
        "Stratified 80/20 split of the filtered official training set; "
        "filtered official test set preserved untouched."
    ),
}

with open(
    OUTPUT_DIR / "split_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2)

print("Saved files:")
for saved_path in sorted(OUTPUT_DIR.iterdir()):
    print("-", saved_path.name)


Saved files:
- class_summary.csv
- split_metadata.json
- test_indices.npy
- train_indices.npy
- validation_indices.npy


## 9. Reusable dataset wrapper

In [35]:
class IndexedCIFARSubset(Dataset):
    def __init__(self, base_dataset, indices, label_map, transform=None):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices)
        self.label_map = dict(label_map)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        base_index = int(self.indices[item])
        image, original_label = self.base_dataset[base_index]
        if self.transform is not None:
            image = self.transform(image)
        return image, self.label_map[original_label]

## 10. Create transforms and DataLoaders

For scientific clarity, the Phase 4 **clean baseline** uses normalization only.

Random cropping, flipping, Random Erasing, and occlusion are intentionally not
applied here. They will be introduced later as separate experimental
conditions so that their effects can be measured fairly.


In [ ]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

# Clean deterministic transform for the baseline experiment.
clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Defined for a later augmentation experiment, but not used in the clean baseline.
augmentation_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_dataset = IndexedCIFARSubset(
    cifar_train_full,
    train_indices,
    old_to_new,
    transform=clean_transform,
)

validation_dataset = IndexedCIFARSubset(
    cifar_train_full,
    validation_indices,
    old_to_new,
    transform=clean_transform,
)

test_dataset = IndexedCIFARSubset(
    cifar_test_full,
    test_indices,
    old_to_new,
    transform=clean_transform,
)

# Fixed generator makes the training shuffle reproducible.
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))

assert images.ndim == 4 and images.shape[1:] == (3, 32, 32)
assert set(torch.unique(labels).tolist()).issubset({0, 1})

print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Unique labels:", torch.unique(labels).tolist())


## 11. Visualize unnormalized samples

In [ ]:
display_dataset = IndexedCIFARSubset(
    cifar_train_full,
    train_indices,
    old_to_new,
    transforms.ToTensor()
)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.reshape(-1)
chosen = random.sample(range(len(display_dataset)), 10)

for ax, i in zip(axes, chosen):
    image, label = display_dataset[i]
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(SELECTED_CLASSES[label])
    ax.axis("off")

plt.suptitle("Fox and Wolf Training Samples")
plt.tight_layout()
plt.show()

# Phase 3 complete

You now have:

- local CIFAR-100 loading with `download=False`;
- a fixed fox/wolf classification task;
- a reproducible stratified 80/20 training-validation split;
- the official fox/wolf test set preserved untouched;
- disjoint split-integrity checks;
- balanced class checks;
- saved split indices and metadata;
- clean, normalized baseline DataLoaders;
- a separate augmentation transform reserved for later experiments.

## Next phase

**Phase 4: Train the clean CNN and ResNet18 baseline models.**

Do not use the augmentation transform during the initial clean-baseline run.


# **Define the simple CNN**

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [37]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [38]:
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [39]:
def evaluate(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    average_loss = running_loss / total
    accuracy = correct / total

    return average_loss, accuracy

In [54]:
def train_model(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    epochs=20
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
    }

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total

        validation_loss, validation_accuracy = evaluate(
            model,
            validation_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: {train_accuracy:.4f} | "
            f"Validation loss: {validation_loss:.4f} | "
            f"Validation accuracy: {validation_accuracy:.4f}"
        )

    return history

In [55]:
evaluate

<function __main__.evaluate(model, data_loader, criterion, device)>

In [56]:
EPOCHS = 20

history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
)

NameError: name 'train_loader' is not defined